# Currency & Economy

Currency flow simulation for Animal Company. Models instance value-in/out, wallet balances, Key Card progression, and Battle Pass economics.

> Reference: `animalco_economy.md` in brain-rag vault

In [1]:
from IPython.display import HTML
display(HTML('''
<style id="aco-toggle-style"></style>
<button id="aco-toggle-btn"
  onclick="
    var style = document.getElementById('aco-toggle-style');
    if (style.innerHTML === '') {
      style.innerHTML = '.jp-Cell-inputWrapper { display: none !important; }';
      this.innerHTML = 'Show Code';
    } else {
      style.innerHTML = '';
      this.innerHTML = 'Hide Code';
    }
  "
  style="padding: 6px 16px; font-size: 13px; cursor: pointer;
         border: 1px solid #ccc; border-radius: 4px; background: #f5f5f5;">
  Hide Code
</button>
'''))

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path

if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

from aco_model.models import EconomyParams, RetentionCurve
from aco_model.retention import load_installs, simulate
from aco_model.economy import simulate_economy
from aco_model.config import load_config
from aco_model.state import load_state, save_state, DEFAULT_STATE_PATH

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

# Load from shared state or config
cfg = load_config()
state = load_state()

if state is not None:
    installs = load_installs(cfg.installs_path)
    curve = RetentionCurve(anchors=state.retention_anchors)
    sim = simulate(installs, curve, state.sim_days)
    econ_params = state.economy or cfg.economy
    print(f'=== Loaded from shared state ({DEFAULT_STATE_PATH}) ===')
    print(f'  Updated: {state.updated_at}')
else:
    installs = load_installs(cfg.installs_path)
    sim = simulate(installs, cfg.retention, cfg.sim_days)
    econ_params = cfg.economy
    print(f'=== No state file — using config.yaml defaults ===')

print(f'  Sim days: {sim.sim_days}')
print(f'  DAU range: {sim.dau.min():,} – {sim.dau.max():,}')
print(f'  Instances/day: {econ_params.instances_per_day}')
print(f'  BP: {econ_params.battle_pass.cost_coins} coins, {econ_params.battle_pass.season_days}-day season')

=== Loaded from shared state (output/state.json) ===
  Updated: 2026-03-25T18:55:46
  Sim days: 90
  DAU range: 15,182 – 53,403
  Instances/day: 3
  BP: 5.0 coins, 60-day season


## 1. Instance Economics

In [3]:
runs_slider = widgets.IntSlider(value=econ_params.instances_per_day, min=1, max=10, step=1,
                                 description='Runs/day:')
buff_cost_slider = widgets.FloatSlider(value=econ_params.buff_cost_scrap, min=0, max=200, step=10,
                                       description='Buff cost:')
bp_rate_slider = widgets.FloatSlider(value=econ_params.battle_pass.purchase_rate * 100,
                                      min=1, max=30, step=1, description='BP buyers %:')

# Store current state context
_anchors = state.retention_anchors if state else cfg.retention.anchors
_monetization = state.monetization if state else cfg.monetization

def _make_econ_params(runs, buff_cost, bp_rate):
    bp = econ_params.battle_pass.model_copy(update={'purchase_rate': bp_rate / 100.0})
    return econ_params.model_copy(update={
        'instances_per_day': runs,
        'buff_cost_scrap': buff_cost,
        'battle_pass': bp,
    })

_slider_args = {'runs': runs_slider, 'buff_cost': buff_cost_slider, 'bp_rate': bp_rate_slider}

def plot_instance_economics(runs, buff_cost, bp_rate):
    params = _make_econ_params(runs, buff_cost, bp_rate)
    econ = simulate_economy(sim, params)
    save_state(sim, _anchors, monetization=_monetization, economy=params)

    ie = econ.instance_economics_dataframe()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    x = np.arange(len(ie))
    width = 0.35
    ax1.bar(x - width/2, ie['value_in'], width, label='Value In', color='#F44336', alpha=0.7)
    ax1.bar(x + width/2, ie['value_out'], width, label='Value Out', color='#4CAF50', alpha=0.7)
    ax1.set_xticks(x)
    ax1.set_xticklabels(ie['tier'])
    ax1.set_ylabel('Value (currency units)')
    ax1.set_title('Value In vs Out per Instance Run')
    ax1.legend()

    ax2.bar(ie['tier'], ie['net_value'], color='#2196F3', alpha=0.7)
    ax2.set_ylabel('Net Value')
    ax2.set_title('Net Value per Instance Run (player profit)')
    ax2.axhline(y=0, color='red', linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.show()

    print(ie.to_string(index=False))

out_ie = widgets.interactive_output(plot_instance_economics, _slider_args)
display(widgets.VBox([
    widgets.HBox([runs_slider, buff_cost_slider, bp_rate_slider]),
    out_ie
]))

## 2. Currency Flows

In [4]:
def plot_currency_flows(runs, buff_cost, bp_rate):
    params = _make_econ_params(runs, buff_cost, bp_rate)
    econ = simulate_economy(sim, params)
    days = np.arange(1, sim.sim_days + 1)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Nuts
    axes[0].fill_between(days, econ.daily_nuts_earned, alpha=0.3, color='#4CAF50', label='Earned')
    axes[0].fill_between(days, econ.daily_nuts_spent, alpha=0.3, color='#F44336', label='Spent')
    axes[0].plot(days, econ.daily_nuts_earned, linewidth=2, color='#4CAF50')
    axes[0].plot(days, econ.daily_nuts_spent, linewidth=2, color='#F44336')
    axes[0].set_xlabel('Day')
    axes[0].set_ylabel('Nuts')
    axes[0].set_title('Daily Nuts: Earned vs Spent')
    axes[0].legend()

    # Scrap
    axes[1].fill_between(days, econ.daily_scrap_earned, alpha=0.3, color='#4CAF50', label='Earned')
    axes[1].fill_between(days, econ.daily_scrap_spent, alpha=0.3, color='#F44336', label='Spent')
    axes[1].plot(days, econ.daily_scrap_earned, linewidth=2, color='#4CAF50')
    axes[1].plot(days, econ.daily_scrap_spent, linewidth=2, color='#F44336')
    axes[1].set_xlabel('Day')
    axes[1].set_ylabel('Scrap')
    axes[1].set_title('Daily Scrap: Earned vs Spent')
    axes[1].legend()

    # Keycards
    axes[2].plot(days, econ.daily_keycards_consumed, linewidth=2, color='#F44336', label='Consumed')
    axes[2].plot(days, econ.daily_keycards_net, linewidth=2, color='#FF9800', label='Net consumed')
    axes[2].set_xlabel('Day')
    axes[2].set_ylabel('Keycards')
    axes[2].set_title('Daily Keycard Consumption')
    axes[2].legend()

    plt.tight_layout()
    plt.show()

out_flows = widgets.interactive_output(plot_currency_flows, _slider_args)
display(out_flows)

Output()

## 3. Wallet Balances

Cumulative currency in the system over time (per GameGou call recommendations).

In [5]:
def plot_wallet_balances(runs, buff_cost, bp_rate):
    params = _make_econ_params(runs, buff_cost, bp_rate)
    econ = simulate_economy(sim, params)
    days = np.arange(1, sim.sim_days + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(days, econ.nuts_balance, linewidth=2, color='#FF9800', label='Nuts')
    ax1.plot(days, econ.scrap_balance, linewidth=2, color='#9C27B0', label='Scrap')
    ax1.set_xlabel('Day')
    ax1.set_ylabel('Cumulative Balance')
    ax1.set_title('Total Currency in System')
    ax1.legend()

    # Per-player average wallet
    dau = sim.dau.astype(float)
    with np.errstate(divide='ignore', invalid='ignore'):
        nuts_per_player = np.where(dau > 0, econ.nuts_balance / dau, 0)
        scrap_per_player = np.where(dau > 0, econ.scrap_balance / dau, 0)

    ax2.plot(days, nuts_per_player, linewidth=2, color='#FF9800', label='Nuts/player')
    ax2.plot(days, scrap_per_player, linewidth=2, color='#9C27B0', label='Scrap/player')
    ax2.set_xlabel('Day')
    ax2.set_ylabel('Avg Balance per Player')
    ax2.set_title('Avg Wallet Balance per Active Player')
    ax2.legend()

    plt.tight_layout()
    plt.show()

    df = econ.to_dataframe()
    print(f'Day {sim.sim_days}: Nuts balance = {df.iloc[-1]["nuts_balance"]:,} | Scrap balance = {df.iloc[-1]["scrap_balance"]:,}')
    if dau[-1] > 0:
        print(f'Avg per player: {nuts_per_player[-1]:,.0f} nuts | {scrap_per_player[-1]:,.0f} scrap')

out_wallet = widgets.interactive_output(plot_wallet_balances, _slider_args)
display(out_wallet)

Output()

## 4. Key Card Progression

In [6]:
def plot_keycard_progression(runs, buff_cost, bp_rate):
    params = _make_econ_params(runs, buff_cost, bp_rate)
    econ = simulate_economy(sim, params)
    kc = econ.keycard_progression_dataframe()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Merge cost escalation
    ax1.bar(kc['tier'], kc['merge_cost_nuts'], color='#FF9800', alpha=0.7)
    ax1.set_ylabel('Nuts')
    ax1.set_title('Merge Cost per Tier')

    # Cumulative investment
    ax2.bar(kc['tier'], kc['cumulative_nuts'], color='#F44336', alpha=0.7, label='Cumulative nuts')
    ax2_r = ax2.twinx()
    ax2_r.plot(kc['tier'], kc['cumulative_cards'], 'o-', color='#2196F3', linewidth=2, label='Cumulative cards')
    ax2.set_ylabel('Nuts', color='#F44336')
    ax2_r.set_ylabel('Cards', color='#2196F3')
    ax2.set_title('Cumulative Investment to Reach Tier')
    ax2.legend(loc='upper left')
    ax2_r.legend(loc='upper right')

    plt.tight_layout()
    plt.show()

    # How many days of nuts earning to afford each tier
    avg_nuts_per_run = np.mean([t.nuts_earned for t in params.instance_tiers])
    nuts_per_day = avg_nuts_per_run * params.instances_per_day
    print(f'Avg nuts earned per day (per player): {nuts_per_day:,.0f}')
    for _, row in kc.iterrows():
        if row['cumulative_nuts'] > 0:
            days_needed = row['cumulative_nuts'] / nuts_per_day
            print(f'  {row["tier"]}: ~{days_needed:.1f} days of play to afford merge costs')

out_kc = widgets.interactive_output(plot_keycard_progression, _slider_args)
display(out_kc)

Output()

## 5. Battle Pass Economics

In [7]:
def plot_battle_pass(runs, buff_cost, bp_rate):
    params = _make_econ_params(runs, buff_cost, bp_rate)
    econ = simulate_economy(sim, params)
    days = np.arange(1, sim.sim_days + 1)
    bp = params.battle_pass

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # BP revenue over time
    cum_bp_rev = np.cumsum(econ.battle_pass_daily_revenue)
    ax1.plot(days, cum_bp_rev, linewidth=2, color='#4CAF50')
    ax1.fill_between(days, cum_bp_rev, alpha=0.2, color='#4CAF50')
    ax1.set_xlabel('Day')
    ax1.set_ylabel('Cumulative BP Revenue (USD)')
    ax1.set_title('Battle Pass Revenue')

    # BP completion rate sensitivity
    rates = [0.1, 0.2, 0.3, 0.5, 0.7]
    for rate in rates:
        bp_var = bp.model_copy(update={'completion_rate': rate})
        params_var = params.model_copy(update={'battle_pass': bp_var})
        econ_var = simulate_economy(sim, params_var)
        # Net coin cost to system = coins in - coins returned to completers
        net_coins = np.cumsum(econ_var.daily_coins_from_bp - econ_var.daily_coins_returned_bp)
        style = {'linewidth': 3, 'linestyle': '--'} if rate == bp.completion_rate else {'linewidth': 2}
        ax2.plot(days, net_coins * params.coin_to_usd, label=f'{rate:.0%} complete', **style)
    ax2.set_xlabel('Day')
    ax2.set_ylabel('Net Coin Revenue (USD)')
    ax2.set_title('BP Net Revenue by Completion Rate')
    ax2.legend()

    plt.tight_layout()
    plt.show()

    bp_buyers_day90 = int(sim.dau[-1] * bp.purchase_rate)
    print(f'Battle Pass: {bp.cost_coins} coins (${bp.cost_coins * params.coin_to_usd:.2f}), {bp.season_days}-day season')
    print(f'BP buyers at day {sim.sim_days}: {bp_buyers_day90:,} ({bp.purchase_rate:.0%} of DAU)')
    print(f'Completion rate: {bp.completion_rate:.0%} — completers get {bp.coins_returned} coins back')
    print(f'Total BP revenue: ${econ.battle_pass_total_revenue:,.2f}')
    print(f'BP rewards per season: {bp.nuts_reward_total:,.0f} nuts, {bp.scrap_reward_total:,.0f} scrap, {bp.keycards_rewarded} keycards')

out_bp = widgets.interactive_output(plot_battle_pass, _slider_args)
display(out_bp)

Output()